In [1]:
import gcsfs
import pandas as pd

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

In [2]:
EXISTING_GCS = "gs://calitp-analytics-data/data-analyses/ntd/"
existing_annual = pd.read_parquet(
    f"{EXISTING_GCS}annual_ridership_report_data.parquet",
    filesystem = gcsfs.GCSFileSystem()
)

In [3]:
def merge_new_df_with_crosswalk(
    filename: str = "annual"
):
    """
    """
    crosswalk = pd.read_parquet(
        f"{GCS_FILE_PATH}crosswalk.parquet", 
        filesystem=gcsfs.GCSFileSystem(),
    ).rename(columns = {"ntd_id_2022": "ntd_id"}).drop_duplicates()

    df = pd.read_parquet(
        f"{GCS_FILE_PATH}{filename}.parquet",
        filesystem=gcsfs.GCSFileSystem(),
    ).merge(
        crosswalk,
        on = "ntd_id",
        how = "left"
    )

    return df

In [4]:
df = merge_new_df_with_crosswalk("annual")

In [5]:
def counts_by_rtpa(
    df: pd.DataFrame,
    group_cols: list
) -> pd.DataFrame:
    """
    Use this to read in existing df vs new annual/monthly df
    and do groupby by rtpa_name or rtpa_name_split,
    and see how counts look overall.
    """
    df2 = (
        df
        .groupby(group_cols, dropna=False)
        .agg(
            total_upt=("upt", "sum"),
            n_agencies=("source_agency", "nunique"),
            agencies=pd.NamedAgg(column="source_agency", aggfunc=lambda x: list(set(x))),
            ntd_ids=pd.NamedAgg(column="ntd_id", aggfunc=lambda x: list(set(x))),
        ).reset_index()
        .astype({"total_upt": "int64"})
        .sort_values(by=group_cols + ["total_upt"], ascending=False)
        .reset_index(drop=True)
    )

    return df2

In [6]:
df2 = counts_by_rtpa(df.rename(columns = {"unlinked_passenger_trips": "upt"}), ["rtpa_name"])

In [7]:
existing_df2 = counts_by_rtpa(existing_annual, ["rtpa_name"])

In [8]:
compare_df = pd.merge(
    existing_df2, 
    df2,
    on = "rtpa_name",
    how = "outer",
    indicator=True
).astype({
    c: "Int64" 
    for c in ["total_upt_x", "total_upt_y", "n_agencies_x", "n_agencies_y"]}
)

In [9]:
compare_df.dtypes

rtpa_name         object
total_upt_x        Int64
n_agencies_x       Int64
agencies_x        object
ntd_ids_x         object
total_upt_y        Int64
n_agencies_y       Int64
agencies_y        object
ntd_ids_y         object
_merge          category
dtype: object

In [10]:
compare_df = compare_df.assign(
    missing_agency = compare_df.apply(
        lambda x:
        list(set([c for c in x.agencies_x if c not in x.agencies_y])) if x._merge=="both"
        else [], 
        axis=1),
    missing_ids = compare_df.apply(
        lambda x:
        list(set([c for c in x.ntd_ids_x if c not in x.ntd_ids_y])) if x._merge=="both"
        else [], 
        axis=1)
)

In [11]:
# bridge is undercounting agencies, which results in fewer upt
results = compare_df[(compare_df._merge=="both") & 
    (compare_df.total_upt_x != compare_df.total_upt_y)]

In [12]:
results

,rtpa_name,total_upt_x,n_agencies_x,agencies_x,ntd_ids_x,total_upt_y,n_agencies_y,agencies_y,ntd_ids_y,_merge,missing_agency,missing_ids
1,Del Norte Local Transportation Commission,84385,2,"[Yurok Tribe - Transportation Department, Elk ...","[99452, 99262]",84077,1,[Yurok Tribe - Transportation Department],[99262],both,[Elk Valley Rancheria (EVR)],[99452]
7,Kings County Association of Governments,28416124,2,"[California Vanpool Authority (CVA), Kings Cou...","[90200, 90230]",4065937,1,[Kings County Area Public Transit Agency (KART)],[90200],both,[California Vanpool Authority (CVA)],[90230]
11,Madera County Transportation Commission,834898,3,[Madera County (MCC) - Public Works Department...,"[90199, 99364, 91005]",824512,2,[Madera County (MCC) - Public Works Department...,"[90199, 91005]",both,[North Fork Rancheria of Mono Indians of Calif...,[99364]
13,Metropolitan Transportation Commission,2331534369,23,"[Napa Valley Transportation Authority (NVTA), ...","[90014, 90078, 90144, 90013, 90234, 90009, 900...",2308006739,20,"[Napa Valley Transportation Authority (NVTA), ...","[90014, 90078, 90144, 90013, 90234, 90009, 900...",both,[San Francisco Bay Area Water Emergency Transp...,"[90089, 90094, 90225]"
17,Sacramento Area Council of Governments,142667186,10,"[Sacramento Regional Transit District, Attenti...","[90167, 90216, 90142, 90090, 90061, 90314, 900...",141160514,6,"[Sacramento Regional Transit District, County ...","[90216, 90142, 90090, 90061, 90019, 90205]",both,[City of Davis (DCT) - Transit/Parks and Commu...,"[90167, 90314, 90220, 90223]"
19,San Diego Association of Governments,546401079,4,"[City of Atascadero - Public Works, San Diego ...","[90030, 90095, 90026, 90194]",54570070,1,[North County Transit District (NCTD)],[90030],both,"[San Diego Metropolitan Transit System (MTS), ...","[90095, 90026, 90194]"
20,San Joaquin Council of Governments,30093684,7,"[City of Tracy - Transit Division, Altamont Co...","[90182, 91078, 90197, 90012, 90217, 90175, 99422]",26534036,6,"[City of Tracy - Transit Division, Altamont Co...","[90182, 91078, 90197, 90012, 90217, 90175]",both,[San Joaquin Council (SJCOG)],[99422]
21,San Luis Obispo Council of Governments,10236333,3,[San Luis Obispo Council of Governments (SLOCO...,"[90156, 90206, 90297]",10159067,2,[San Luis Obispo Regional Transit Authority (S...,"[90156, 90206]",both,[San Luis Obispo Council of Governments (SLOCOG)],[90297]
22,Santa Barbara County Association of Governments,38826821,5,[Santa Barbara Metropolitan Transit District (...,"[90243, 90087, 90303, 90020, 90149]",38387059,4,[Santa Barbara Metropolitan Transit District (...,"[90303, 90020, 90087, 90149]",both,[Easy Lift Transportation],[90243]
26,Stanislaus Council of Governments,18476060,5,"[Stanislaus Regional Transit Authority, Stanis...","[90201, 90311, 90236, 90306, 90007]",8924915,2,"[City of Turlock - Transit, Stanislaus Regiona...","[90201, 90306]",both,"[Stanislaus County (StaRT), City of Modesto (M...","[90007, 90311, 90236]"


In [14]:
for rtpa in results.rtpa_name.unique():
    print(rtpa)
    subset_df = results[results.rtpa_name==rtpa].reset_index(drop=True)
    print(subset_df.missing_agency.iloc[0])
    print(subset_df.missing_ids.iloc[0])
    print("******************************************************************")

Del Norte Local Transportation Commission
['Elk Valley Rancheria (EVR)']
['99452']
******************************************************************
Kings County Association of Governments
['California Vanpool Authority (CVA)']
['90230']
******************************************************************
Madera County Transportation Commission
['North Fork Rancheria of Mono Indians of California (NFR) - Administration Department']
['99364']
******************************************************************
Metropolitan Transportation Commission
['San Francisco Bay Area Water Emergency Transportation Authority (WETA)', 'Metropolitan Transportation Commission (MTC) - Field Operations and Asset Management', 'County of Sonoma (SCT) - Department of Public Infrastructure - Transit Division']
['90089', '90094', '90225']
******************************************************************
Sacramento Area Council of Governments
['City of Davis (DCT) - Transit/Parks and Community Services', 'Atten